In [ ]:
pip install openai


In [9]:
# Detect dominant emotions in tests/images using the OpenAI vision API
# Installs: pip install openai
# API key: set OPENAI_API_KEY env var, store it in a .env file, or enter interactively when prompted.
import base64
import json
import os
from pathlib import Path
from getpass import getpass

from openai import OpenAI


def encode_image_bytes(path: Path) -> str:
    """Return the file as a base64 data URL payload."""
    return base64.b64encode(path.read_bytes()).decode("utf-8")


def load_api_key_from_env_file() -> str | None:
    env_paths = [
        Path(".env"),
        Path("..") / ".env",
        Path("tests") / ".env",
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
    ]
    for env_path in env_paths:
        if not env_path.exists():
            continue
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line.split(" ", 1)[1].strip()
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            if key.strip() == "OPENAI_API_KEY":
                return value.strip().strip('"').strip("'")
    return None


image_dir_candidates = [
    Path("images"),
    Path("tests") / "images",
    Path.cwd() / "tests" / "images",
]
image_dir = next((p for p in image_dir_candidates if p.exists()), None)
if image_dir is None:
    raise FileNotFoundError(
        "Could not locate the images folder (expected at ./images or ./tests/images)."
    )

api_key = os.environ.get("OPENAI_API_KEY") or load_api_key_from_env_file()
if not api_key:
    api_key = getpass("Enter OPENAI_API_KEY: ").strip()
    if not api_key:
        raise RuntimeError("An OpenAI API key is required to run this cell.")

client = OpenAI(api_key=api_key)

allowed_suffixes = {".png", ".jpg", ".jpeg", ".webp"}
emotion_guidelines = (
    "Identify the dominant visible emotion of the person in the photo. "
    "Pick one label from ['happy','sad','angry','surprised','fearful','disgusted','neutral']. "
    "If you cannot tell, use 'uncertain'. Respond ONLY with JSON in the form "
    "{\"emotion_label\": <label>, \"confidence\": <0-1 float>, \"rationale\": <visual cues>}"
)

results = []
paths = [
    p
    for p in image_dir.iterdir()
    if p.is_file() and p.suffix.lower() in allowed_suffixes
]
for image_path in sorted(paths, key=lambda p: p.name):
    response = client.responses.create(
        model="gpt-5-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"{emotion_guidelines} File name: {image_path.name}",
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/{image_path.suffix.lstrip('.').lower()};base64,{encode_image_bytes(image_path)}",
                    },
                ],
            }
        ],
        max_output_tokens=300,
    )
    output_text = getattr(response, "output_text", None)
    if output_text is None:
        output_text = json.dumps(response.model_dump(), ensure_ascii=False)
    try:
        parsed = json.loads(output_text)
    except json.JSONDecodeError:
        parsed = {"emotion_label": "unparsed", "rationale": output_text}
    parsed["filename"] = image_path.name
    results.append(parsed)

print("Detected emotions:")

for entry in results:
    emotion = entry.get("emotion_label", "unknown")
    conf = entry.get("confidence")
    rationale = entry.get("rationale") or entry.get("raw_response", "")
    conf_txt = ""
    if conf is not None:
        try:
            conf_txt = f" (confidence={float(conf):.2f})"
        except (TypeError, ValueError):
            conf_txt = f" (confidence={conf})"
    print(f"- {entry['filename']}: {emotion}{conf_txt} -> {rationale}")

print("Raw JSON response:")
print(json.dumps(results, indent=2, ensure_ascii=False))


Detected emotions:
- human1.jpeg: neutral (confidence=0.85) -> Mouth is closed with a straight line, jaw and facial muscles appear relaxed, eyes are not widened and eyebrows are not raised or deeply furrowed — overall a neutral, expressionless appearance.
- human2.jpeg: happy (confidence=0.95) -> Broad smile with visible teeth, upturned mouth corners, slight eye narrowing (crow's feet), and relaxed, open facial expression consistent with joy.
- human3.jpeg: angry (confidence=0.90) -> Wide open mouth with visible teeth, clenched jaw, furrowed brows, narrowed eyes and tense facial muscles suggest shouting/anger while holding a phone.
Raw JSON response:
[
  {
    "emotion_label": "neutral",
    "confidence": 0.85,
    "rationale": "Mouth is closed with a straight line, jaw and facial muscles appear relaxed, eyes are not widened and eyebrows are not raised or deeply furrowed — overall a neutral, expressionless appearance.",
    "filename": "human1.jpeg"
  },
  {
    "emotion_label": "happy

In [10]:
"""
Batch emotion detection on images using the OpenAI Vision-capable model.

- Reads images from ./images or ./tests/images
- Loads OPENAI_API_KEY from environment, .env, or interactive prompt
- Calls the OpenAI Responses API once per image
- Prints a per-file summary and raw JSON list

Outputs (per image):
{
  "filename": "<name>",
  "emotion_label": "<happy|sad|angry|surprised|fearful|disgusted|neutral|uncertain|unparsed>",
  "confidence": <0..1 float | optional>,
  "rationale": "<short visual cues or raw text>"
}
"""

import base64
import json
import os
from dataclasses import dataclass
from pathlib import Path
from getpass import getpass
from typing import Any

from openai import OpenAI


@dataclass(frozen=True)
class AppConfig:
    """Application configuration constants."""
    model_name: str = "gpt-5-mini"
    max_output_tokens: int = 300
    allowed_suffixes: frozenset[str] = frozenset({".png", ".jpg", ".jpeg", ".webp"})
    image_dir_candidates: tuple[Path, ...] = (
        Path("images"),
        Path("tests") / "images",
        Path.cwd() / "tests" / "images",
    )
    emotion_guidelines: str = (
        "Identify the dominant visible emotion of the person in the photo. "
        "Pick one label from ['happy','sad','angry','surprised','fearful','disgusted','neutral']. "
        "If you cannot tell, use 'uncertain'. Respond ONLY with JSON in the form "
        "{\"emotion_label\": <label>, \"confidence\": <0-1 float>, \"rationale\": <visual cues>}"
    )


def encode_image_bytes(path: Path) -> str:
    """
    Read a file and return its base64-encoded contents (no prefix).

    Args:
        path: Path to the image file.

    Returns:
        Base64-encoded string.
    """
    return base64.b64encode(path.read_bytes()).decode("utf-8")


def load_api_key_from_env_file() -> str | None:
    """
    Load OPENAI_API_KEY from a .env-style file in common locations.

    Returns:
        API key string if found, else None.
    """
    env_paths = (
        Path(".env"),
        Path("..") / ".env",
        Path("tests") / ".env",
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
    )

    for env_path in env_paths:
        if not env_path.exists():
            continue

        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue

            if line.startswith("export "):
                line = line.split(" ", 1)[1].strip()

            if "=" not in line:
                continue

            key, value = line.split("=", 1)
            if key.strip() == "OPENAI_API_KEY":
                return value.strip().strip('"').strip("'")

    return None


def resolve_image_dir(config: AppConfig) -> Path:
    """
    Resolve the input image directory.

    Args:
        config: Application configuration.

    Returns:
        Path to the images directory.

    Raises:
        FileNotFoundError: If no expected directory exists.
    """
    image_dir = next((p for p in config.image_dir_candidates if p.exists()), None)
    if image_dir is None:
        raise FileNotFoundError(
            "Could not locate the images folder (expected at ./images or ./tests/images)."
        )
    return image_dir


def resolve_api_key() -> str:
    """
    Resolve the OpenAI API key from environment, .env, or interactive prompt.

    Returns:
        API key string.

    Raises:
        RuntimeError: If no API key is provided.
    """
    api_key = os.environ.get("OPENAI_API_KEY") or load_api_key_from_env_file()
    if api_key:
        return api_key

    api_key = getpass("Enter OPENAI_API_KEY: ").strip()
    if not api_key:
        raise RuntimeError("An OpenAI API key is required to run this script.")
    return api_key


def build_data_url(image_path: Path) -> str:
    """
    Build a data URL for the given image file.

    Args:
        image_path: Path to the image file.

    Returns:
        data:image/<suffix>;base64,<payload>
    """
    suffix = image_path.suffix.lstrip(".").lower()
    payload = encode_image_bytes(image_path)
    return f"data:image/{suffix};base64,{payload}"


def safe_parse_model_json(output_text: str) -> dict[str, Any]:
    """
    Parse model output as JSON. If parsing fails, return an 'unparsed' payload.

    Args:
        output_text: Model output text.

    Returns:
        Parsed JSON object or a fallback dict.
    """
    try:
        parsed = json.loads(output_text)
        if isinstance(parsed, dict):
            return parsed
        return {"emotion_label": "unparsed", "rationale": output_text}
    except json.JSONDecodeError:
        return {"emotion_label": "unparsed", "rationale": output_text}


def detect_emotion_for_image(
    client: OpenAI, config: AppConfig, image_path: Path
) -> dict[str, Any]:
    """
    Run emotion detection for a single image.

    Args:
        client: OpenAI client.
        config: Application configuration.
        image_path: Path to the image file.

    Returns:
        Result dict containing at minimum 'filename' and 'emotion_label'.
    """
    data_url = build_data_url(image_path)

    response = client.responses.create(
        model=config.model_name,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": f"{config.emotion_guidelines} File name: {image_path.name}",
                    },
                    {"type": "input_image", "image_url": data_url},
                ],
            }
        ],
        max_output_tokens=config.max_output_tokens,
    )

    output_text = getattr(response, "output_text", None)
    if output_text is None:
        output_text = json.dumps(response.model_dump(), ensure_ascii=False)

    parsed = safe_parse_model_json(output_text)
    parsed["filename"] = image_path.name
    return parsed


def list_images(config: AppConfig, image_dir: Path) -> list[Path]:
    """
    List supported image files in the directory.

    Args:
        config: Application configuration.
        image_dir: Directory to scan.

    Returns:
        Sorted list of image file paths.
    """
    images = [
        p
        for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in config.allowed_suffixes
    ]
    return sorted(images, key=lambda p: p.name)


def print_summary(results: list[dict[str, Any]]) -> None:
    """
    Print a human-readable summary to stdout.

    Args:
        results: List of per-image result dicts.
    """
    print("Detected emotions:")
    for entry in results:
        filename = entry.get("filename", "<unknown>")
        emotion = entry.get("emotion_label", "unknown")

        conf = entry.get("confidence")
        conf_txt = ""
        if conf is not None:
            try:
                conf_txt = f" (confidence={float(conf):.2f})"
            except (TypeError, ValueError):
                conf_txt = f" (confidence={conf})"

        rationale = entry.get("rationale") or entry.get("raw_response", "")
        print(f"- {filename}: {emotion}{conf_txt} -> {rationale}")


def main() -> None:
    """Program entry point."""
    config = AppConfig()

    image_dir = resolve_image_dir(config)
    api_key = resolve_api_key()
    client = OpenAI(api_key=api_key)

    images = list_images(config, image_dir)

    results: list[dict[str, Any]] = []
    for image_path in images:
        result = detect_emotion_for_image(client, config, image_path)
        results.append(result)

    print_summary(results)

    print("Raw JSON response:")
    print(json.dumps(results, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


Detected emotions:
- human1.jpeg: neutral (confidence=0.80) -> Face shows a relaxed, closed mouth with no smile or pronounced frown, eyes and brows are not widely opened or sharply furrowed, and there are no visible signs of surprise, disgust, fear or anger—overall a neutral expression.
- human2.jpeg: happy (confidence=0.95) -> Broad smile with visible teeth, raised cheeks causing slight eye crinkling, relaxed facial muscles and open, bright expression indicating positive emotion.
- human3.jpeg: angry (confidence=0.92) -> Wide open mouth with visible teeth (yelling), furrowed/knitted brows, squinted eyes and tense facial muscles while holding a phone — classic signs of anger/hostility.
Raw JSON response:
[
  {
    "emotion_label": "neutral",
    "confidence": 0.8,
    "rationale": "Face shows a relaxed, closed mouth with no smile or pronounced frown, eyes and brows are not widely opened or sharply furrowed, and there are no visible signs of surprise, disgust, fear or anger—overall a ne